In [ ]:
# 矢板の水平変位プロファイル
# 端断面は Y_target = 18.0 m、許容ずれ tolerance = 0.01 m。
# 壁の途中を見るときは、節点がその Y の近くに並ぶ幅まで tolerance を広げます。
# 曲線が細かくなりすぎたら tolerance を狭めます。
Y_target = 18.0  # m
tolerance = 0.01  # m


## 手順

1. Colab では、先に `sheetpile-deformation-viz準備.ipynb` を実行し、Google Drive の Colab Notebooks にリポジトリを置きます。
2. PLAXIS の全変位表（TXT）を `data/<断面名>/` に置きます。例は `data/landside` と `data/riverfront` です。このデータは GitHub に含まれません。
3. 上のセルで目標の Y と許容ずれを決め、このノートのセルを上から実行します。
4. `data` 直下のフォルダごとに 1 枚の図を描き、`figures/` に保存します。採用点が 0 件の段階は描かず、警告します。


In [ ]:
from pathlib import Path
import sys

REPO_NAME = "sheetpile-deformation-viz"
COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks") / REPO_NAME

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_root(start: Path):
    """src/sheetpile_profile.py があるディレクトリを返す。"""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "sheetpile_profile.py").is_file():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = COLAB_ROOT
    if not (ROOT / "src" / "sheetpile_profile.py").is_file():
        raise FileNotFoundError(
            "リポジトリがありません。先に sheetpile-deformation-viz準備.ipynb を実行してください。"
        )
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise FileNotFoundError(
            "src/sheetpile_profile.py が見つかりません。リポジトリの中で開いてください。"
        )

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA = ROOT / "data"
FIGURES = ROOT / "figures"
print(f"ROOT = {ROOT}")


In [ ]:
import matplotlib.pyplot as plt
from sheetpile_profile import (
    configure_japanese_font,
    list_displacement_files,
    plot_folder,
    series_warnings,
)

print(f"フォント: {configure_japanese_font()}")

if not DATA.is_dir():
    raise FileNotFoundError(
        "data フォルダがありません。PLAXIS の全変位表を data/<断面名>/*.TXT として置いてください。"
        " 例: data/landside と data/riverfront"
    )

CAPTIONS = {"landside": "陸側", "riverfront": "川表"}
folders = sorted(
    path for path in DATA.iterdir() if path.is_dir() and not path.name.startswith(".")
)
if not folders:
    raise FileNotFoundError("data の直下に断面フォルダがありません。")

FIGURES.mkdir(parents=True, exist_ok=True)

for folder in folders:
    caption = CAPTIONS.get(folder.name, folder.name)
    if not list_displacement_files(folder):
        print(f"警告: {caption} に TXT がないため、このフォルダは描きません。")
        continue
    output = FIGURES / f"{folder.name}.png"
    _fig, series = plot_folder(
        folder,
        Y_target,
        tolerance,
        output_path=output,
        title=f"{caption}（Y = {Y_target:g} m、許容ずれ {tolerance:g} m）",
    )
    print(caption)
    for item in series:
        print(f"  {item.label}: {item.n_points} 点")
    for message in series_warnings(series):
        print(message)
    print(f"保存: {output}")

plt.show()
